# DAY 2 - 모델 개발 및 평가

Batch 1로 모델을 개발하고, 충전 프로토콜 단위 nested CV와 hold-out을 거쳐 Batch 2를 한 번만 최종 평가한다. knee point, 후기 slope, 최종 수명 기반 비율은 사용하지 않는다.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_CANDIDATES = [
    CURRENT_DIR,
    CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else None,
]
PROJECT_ROOT = next(
    (path for path in PROJECT_CANDIDATES if path is not None and (path / 'src' / 'train.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')
RESULTS_DIR = PROJECT_ROOT / 'results'
DATA_DIR = PROJECT_ROOT / 'data'
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

## 전체 파이프라인 재실행

원본 MAT 파일이 `data/`에 있을 때 아래 값을 `True`로 바꾸면 피처 추출부터 다시 실행한다. 원본 용량은 약 8GB다.

In [2]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    import sys
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.train import run_project
    selected = run_project(DATA_DIR, RESULTS_DIR, rebuild_features=True)
    display(selected)

## 데이터 및 누수 점검

In [3]:
features = pd.read_csv(RESULTS_DIR / 'feature_table.csv')
display(features.groupby('batch').agg(cells=('cell_key', 'size'), labeled=('cycle_life', 'count'), mean_life=('cycle_life', 'mean')))
display(features.select_dtypes('number').corr()['cycle_life'].drop('cycle_life').sort_values(key=abs, ascending=False).head(12).to_frame('pearson_r'))

,cells,labeled,mean_life
batch,,,
Batch 1,46,46,844.717
Batch 2,47,39,565.744
Batch 3,46,44,"1,059.659"


,pearson_r
dq_min,0.827
dq_std,-0.824
dq_range,-0.800
dq_mean,0.797
initial_QDischarge,-0.525
QDischarge_mean_100,-0.500
dq_kurtosis,0.365
temperature_slope_100,0.345
IR_mean_100,-0.336
max_c_rate,-0.260


전처리와 하이퍼파라미터 탐색은 각 training fold 안에서 수행한다. Batch 1 hold-out에는 개발 구간과 같은 `policy_readable` 그룹이 하나도 없다.

In [4]:
split = pd.read_csv(RESULTS_DIR / 'tables' / 'split_manifest.csv')
dev_protocols = set(split.loc[split['split'].eq('Development train'), 'policy_readable'])
valid_protocols = set(split.loc[split['split'].eq('Batch 1 hold-out'), 'policy_readable'])
print('protocol overlap:', sorted(dev_protocols & valid_protocols))
display(split.groupby('split').agg(cells=('cell_key', 'size'), protocols=('policy_readable', 'nunique')))

protocol overlap: []


,cells,protocols
split,,
Batch 1 hold-out,13,6
Development train,33,17


## 후보 모델과 ablation

In [5]:
comparison = pd.read_csv(RESULTS_DIR / 'tables' / 'model_comparison.csv')
best_by_family = comparison.loc[comparison.groupby('model')['cv_mape_percent'].idxmin()]
display(best_by_family.sort_values('cv_mape_percent')[['model', 'target_transform', 'feature_set', 'cv_mape_percent', 'cv_std_percent', 'holdout_mape_percent']])
ablation = pd.read_csv(RESULTS_DIR / 'tables' / 'feature_ablation.csv')
display(ablation[['feature_set', 'cv_mape_percent', 'cv_std_percent', 'holdout_mape_percent']])

,model,target_transform,feature_set,cv_mape_percent,cv_std_percent,holdout_mape_percent
0,ElasticNet,log,base_initial_q,7.890,4.482,7.774
2,Ridge,raw,base_initial_q,9.407,5.777,11.114
8,Linear,log,base_initial_q,10.330,2.460,9.376
10,RandomForest,log,base_initial_q,11.778,1.965,12.089
14,HistGradientBoosting,log,base_mean_q,13.780,1.729,11.647
20,Dummy,raw,base_initial_q,22.039,5.465,14.203


,feature_set,cv_mape_percent,cv_std_percent,holdout_mape_percent
0,dq_std_substitution,7.617,4.016,7.965
1,no_protocol,7.837,4.535,7.774
2,no_dq_shape,7.838,4.554,7.496
3,base_initial_q,7.890,4.482,7.774
4,dq_range_substitution,7.892,4.388,7.921
5,dq_only,8.936,3.816,7.682
6,base_mean_q,8.987,3.662,9.216
7,dq_mean_substitution,10.058,4.027,8.274


## 요구 성능표와 Gap 해석

In [6]:
performance = pd.read_csv(RESULTS_DIR / 'model_performance.csv')
display(performance.style.format({'mape_percent': '{:.2f}'}))
metadata = json.loads((RESULTS_DIR / 'tables' / 'run_metadata.json').read_text())
display(metadata)

,index,mape_percent,note
0,Train (Batch 1 CV),7.62,Development partition; protocol-grouped nested CV mean
1,Valid (Batch 1 Hold-out),7.96,Unseen charging protocols; model fit on development partition
2,Test (Batch 2),37.46,External batch; selected model refit on all Batch 1
3,Gap (Train-Valid),0.35,(+) indicates possible overfitting
4,Gap (Valid-Test),29.50,(+) indicates batch generalization degradation
5,Gap (Target-Test),28.36,Paper target: 9.1% MAPE


{'random_state': 42,
 'selected_model': 'ElasticNet',
 'selected_target_transform': 'log',
 'selected_feature_set': 'dq_std_substitution',
 'selected_features': ['dq_std',
  'dq_kurtosis',
  'initial_QDischarge',
  'IR_mean_100',
  'temperature_slope_100',
  'chargetime_mean_100',
  'max_c_rate'],
 'selection_rule': 'Lowest Batch 1 nested group-CV mean MAPE; CV dispersion and model simplicity are tie-breakers. Feature set is then selected by the same nested group-CV criterion.',
 'holdout_used_for_selection': False,
 'batch2_used_for_selection': False,
 'final_best_params': {'regressor__model__alpha': 0.01,
  'regressor__model__l1_ratio': 0.9},
 'sample_counts': {'batch1': 46,
  'development': 33,
  'holdout': 13,
  'batch2': 39,
  'batch3_labeled': 44,
  'batch3_paper_screened': 40},
 'batch3_paper_excluded_one_based_cell_numbers': [3, 24, 33, 38, 43, 44],
 'protocol_overlap_development_holdout': [],
 'paper_target_mape_percent': 9.1}

Train-Valid Gap은 작지만 Valid-Test Gap은 크다. 이는 내부 과적합보다 Batch 2의 단수명 집중과 unseen protocol에 의한 배치 일반화 문제가 지배적이라는 뜻이다.

## 오류 분석 및 Batch 3 추가 검증

In [7]:
errors = pd.read_csv(RESULTS_DIR / 'tables' / 'predictions_and_errors.csv')
display(pd.read_csv(RESULTS_DIR / 'tables' / 'error_analysis_summary.csv'))
display(pd.read_csv(RESULTS_DIR / 'tables' / 'batch2_error_by_life_group.csv'))
display(errors.loc[errors['evaluation_set'].eq('Test (Batch 2)')].nlargest(10, 'absolute_percentage_error'))
display(pd.read_csv(RESULTS_DIR / 'tables' / 'supplementary_metrics.csv'))

,evaluation_set,n_cells,mape_percent,median_ape_percent,mean_signed_error_cycles,overprediction_rate_percent
0,Optional Test (Batch 3),44,14.501,10.635,-174.049,20.455
1,Test (Batch 2),39,37.462,43.534,179.100,94.872
2,Valid (Batch 1 Hold-out),13,7.965,6.288,19.955,53.846


,life_group,n_cells,mape_percent,mean_signed_error_cycles,overprediction_rate_percent
0,Short,28,40.392,181.395,100.000
1,Middle,8,36.786,261.955,100.000
2,Long,3,11.915,-63.263,33.333


,cell_key,batch,policy_readable,cycle_life,evaluation_set,prediction,signed_error,absolute_percentage_error
44,2018-02-20_cell_7,Batch 2,3.6C(9%)-5C,393.000,Test (Batch 2),690.990,297.990,75.824
45,2018-02-20_cell_19,Batch 2,5.2C(50%)-4.25C,449.000,Test (Batch 2),736.317,287.317,63.990
46,2018-02-20_cell_30,Batch 2,5.2C(58%)-4C,452.000,Test (Batch 2),740.658,288.658,63.862
47,2018-02-20_cell_12,Batch 2,5.2C(50%)-4.25C,449.000,Test (Batch 2),693.813,244.813,54.524
48,2018-02-20_cell_47,Batch 2,4.8C(80%)-4.8C,503.000,Test (Batch 2),777.249,274.249,54.523
49,2018-02-20_cell_29,Batch 2,4.8C(80%)-4.8C,437.000,Test (Batch 2),672.117,235.117,53.802
50,2018-02-20_cell_16,Batch 2,3.6C(9%)-5C,396.000,Test (Batch 2),608.801,212.801,53.738
51,2018-02-20_cell_43,Batch 2,5.2C(50%)-4.25C,474.000,Test (Batch 2),719.993,245.993,51.897
52,2018-02-20_cell_1,Batch 2,5.2C(58%)-4C,477.000,Test (Batch 2),721.215,244.215,51.198
53,2018-02-20_cell_25,Batch 2,4.8C(80%)-4.8C,514.000,Test (Batch 2),775.163,261.163,50.810


,set,mape_percent,mae_cycles,rmse_cycles,r2
0,Valid (Batch 1 Hold-out),7.965,66.370,78.919,0.796
1,Test (Batch 2),37.462,194.243,212.721,0.059
2,"Optional Test (Batch 3, all labeled cells)",14.501,189.529,296.930,0.084
3,"Optional Test (Batch 3, paper-screened)",13.671,175.290,284.555,0.128
